In [2]:
%matplotlib inline
import os
import pandas as pd
import sionna.rt
import matplotlib.pyplot as plt
import mitsuba as mi
import drjit as dr
import numpy as np
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, RadioMapSolver
from pathlib import Path 

In [3]:
SCENE_FILENAME = Path('blender/massy.xml')
FREQUENCY = 3500  # Only supports: 700, 800, 900, 1800, 2100, 2600, 3500
FREQUENCY_UNIT='mhz' # Only supports: mhz, ghz
TRANSMITTER_DIRECTORY = Path('data/transimitters')
RECEIVER_FILENAME = Path('data/sensor_location.csv')

In [12]:
# Load the Mitsuba description scene exported from Blender
scene = sionna.rt.load_scene(filename=SCENE_FILENAME)

# Configure antenna arrays for transmitters (Tx) and receivers (Rx)
tx_array1 = sionna.rt.PlanarArray(
    num_rows=2,
    num_cols=2,
    vertical_spacing=0.5,
    horizontal_spacing=0.5,
    pattern="tr38901",
    polarization="cross",
    polarization_model="tr38901_2")
tx_array2 = sionna.rt.PlanarArray(
    num_rows=1,
    num_cols=1,
    vertical_spacing=0.5,
    horizontal_spacing=0.5,
    pattern="tr38901",
    polarization="cross")
rx_array = sionna.rt.PlanarArray(
    num_rows=1,
    num_cols=1,
    vertical_spacing=0.5,
    horizontal_spacing=0.5,
    pattern="dipole",
    polarization="cross")

scene.tx_array = tx_array1 if FREQUENCY == 3500 and FREQUENCY_UNIT=='mhz' else tx_array2
scene.rx_array = rx_array

In [ ]:
from pyproj import Transformer, CRS

# Define the geographical boundary of the target region
LAT_MAX, LAT_MIN = 48.7409, 48.7171
LON_MIN, LON_MAX = 2.2451, 2.3013

# Calculate the center origin point of the scene
# LAT_ORIGIN = (LAT_MAX + LAT_MIN)/2
# LON_ORIGIN = (LON_MIN + LON_MAX)/2
LAT_ORIGIN = (LAT_MAX + LAT_MIN)/2
LON_ORIGIN = (LON_MIN + LON_MAX)/2
ALT_ORIGIN = 0

DEFAULT_AZIMUTH_MUTIPLITER=1
DEFAULT_OFF_SET=0

# Determine the corresponding UTM (Universal Transverse Mercator) zone based on longitude
_ZONE = int((LON_ORIGIN + 180) / 6) + 1
_IS_SOUTH = LAT_ORIGIN < 0
UTM_CRS = CRS.from_dict({
        'proj': 'utm', 
        'zone': _ZONE,
        'south': _IS_SOUTH
    })

def latlonh_to_xyz_batch(
    lat: pd.Series | float, lon: pd.Series | float, h: pd.Series | float
) -> tuple[np.ndarray | float, np.ndarray | float, np.ndarray | float]:
    """
    Batch convert geographic coordinates (Latitude, Longitude, height) to
    the local Cartesian coordinates (x, y, z) of the scene.

    Returns np.ndarray if inputs are Pandas Series, otherwise returns float.
    """
    transformer = Transformer.from_crs("epsg:4326", UTM_CRS, always_xy=True)
    x_origin, y_origin = transformer.transform(LON_ORIGIN, LAT_ORIGIN)

    # Convert potential Series inputs to NumPy arrays to ensure consistent output type
    lat_arr = np.asarray(lat)
    lon_arr = np.asarray(lon)
    h_arr = np.asarray(h)

    # pyproj transform natively accepts NumPy arrays
    x_objs, y_objs = transformer.transform(lon_arr, lat_arr)

    # Calculate relative coordinates based on the scene's origin center
    xs = x_objs - x_origin
    ys = y_objs - y_origin
    zs = h_arr - ALT_ORIGIN

    # If the original input was a float, extract the scalar from the 0-d/1-d array
    if isinstance(lat, float) or isinstance(lat, int):
        return float(xs), float(ys), float(zs)

    # Otherwise, ensure it returns a standard NumPy array
    return np.asarray(xs), np.asarray(ys), np.asarray(zs)


def deflected_azimuth(
    azimuth: pd.Series | float,
    multiplier: float = DEFAULT_AZIMUTH_MUTIPLITER,
    offset: float = DEFAULT_OFF_SET,
) -> np.ndarray | float:
    """
    Apply multiplication and addition to the input azimuth.

    Returns np.ndarray if azimuth is a Pandas Series, otherwise returns float.
    """
    # If the input is a single float, perform scalar math and return a float
    if isinstance(azimuth, (float, int)):
        return azimuth * multiplier + offset

    # If the input is a Pandas Series (or any array-like), convert it to
    # a NumPy array first to ensure the return type is np.ndarray.
    az_arr = np.asarray(azimuth)

    return az_arr * multiplier + offset

In [6]:
DISPLAY_RADIUS=None
RECEIVER_GROUND_HEIGHT=4
DEFAULT_TRANSIMITTER_DBM=40
DEFAULT_PITCH=0
DEFAULT_ROLL=0

def get_tx_name(id):
    return f'tx_{id}'

def get_rx_name(id):
    return f'rx_{id}'

def add_tx(scene, name, position, orientation):
    tx = Transmitter(
        name=name, 
        position=position, 
        orientation=orientation, 
        power_dbm=DEFAULT_TRANSIMITTER_DBM,
        display_radius=DISPLAY_RADIUS)
    scene.add(tx)

def add_txs(scene, df_tx: pd.DataFrame):
    """
    Batch map geographic Tx coordinates into the 3D scene and instantiate
    Transmitter objects.
    """
    xs, ys, zs = latlonh_to_xyz_batch(
        df_tx["Latitude"], df_tx["Longitude"], df_tx["height"]
    )

    azimuths = deflected_azimuth(df_tx["Azimut"])

    for x, y, z, azimuth, row in zip(
        xs, ys, zs, azimuths, df_tx.itertuples(index=False)
    ):
        name = get_tx_name(row.ID)

        # Debug output to verify transmitter height alignment
        print(f"TX {name} placed at x={x:.2f}, y={y:.2f}, z={z:.2f}, azimuth={azimuth:.2f}")

        # Explicitly cast to Python native float as expected by the scene renderer
        add_tx(
            scene,
            name,
            (float(x), float(y), float(z)),
            (float(azimuth), DEFAULT_PITCH, DEFAULT_ROLL)
        )

def add_rx(scene, name, position):
    rx = Receiver(name=name, position=position, display_radius=DISPLAY_RADIUS)
    scene.add(rx)

def add_rxs(scene, df_rx: pd.DataFrame):
    """
    Batch map geographic Rx coordinates into the 3D scene and instantiate
    Receiver objects.
    """
    xs, ys, z = latlonh_to_xyz_batch(
        df_rx["Latitude"], df_rx["Longitude"], RECEIVER_GROUND_HEIGHT
    )  # All receivers are at the same height.

    z_float = float(z)  # All receivers are at the same height.

    for x, y, row in zip(xs, ys, df_rx.itertuples(index=False)):
        name = get_rx_name(row.sensor_id)

        # Debug output to verify sensor height alignment
        print(f"Sensor {name} placed at x={x:.2f}, y={y:.2f}, z={z:.2f}")

        # Explicitly cast to standard Python floats for the scene renderer
        add_rx(scene, name, (float(x), float(y), z_float))

In [7]:
def set_frequence(scene, frequence, unit:str = FREQUENCY_UNIT):
    """
    Utility function to configure the carrier frequency of the simulation scene.
    """
    if unit.lower() == 'mhz':
        scene.frequency = frequence * 1e6
    elif unit.lower() == 'ghz':
        scene.frequency = frequence * 1e9
    else:
        raise ValueError(f"Unsupported frequency unit: {unit}")

In [8]:
# Load the preprocessed transmitter and receiver information from files
transmitter_filename = TRANSMITTER_DIRECTORY / f'{FREQUENCY}_mhz.csv'

df_tx = pd.read_csv(transmitter_filename, encoding='utf-8-sig')
df_rx = pd.read_csv(RECEIVER_FILENAME, encoding='utf-8')

In [14]:
# TODO: Some ITU materials are currently undefined at specific custom frequencies.
# Therefore, we temporarily fall back to Sionna's default band setup.
# Once parameters are complete, uncomment the line below to fully enable custom frequencies:
set_frequence(scene, FREQUENCY, FREQUENCY_UNIT)
add_txs(scene, df_tx)
add_rxs(scene, df_rx)

TX tx_1 placed at x=-1074.18, y=228.52, z=41.50, azimuth=254.00
TX tx_2 placed at x=-1076.58, y=232.84, z=41.50, azimuth=17.00
TX tx_3 placed at x=-1068.95, y=233.77, z=41.50, azimuth=120.00
TX tx_4 placed at x=-706.36, y=-1186.82, z=22.00, azimuth=240.00
TX tx_5 placed at x=-706.36, y=-1186.82, z=22.00, azimuth=0.00
TX tx_6 placed at x=-706.36, y=-1186.82, z=22.00, azimuth=120.00
TX tx_7 placed at x=-976.52, y=913.20, z=50.20, azimuth=0.00
TX tx_8 placed at x=-966.06, y=898.45, z=50.20, azimuth=120.00
TX tx_9 placed at x=-982.45, y=893.07, z=50.20, azimuth=240.00
TX tx_10 placed at x=-975.30, y=901.58, z=48.80, azimuth=30.00
TX tx_11 placed at x=-975.30, y=901.58, z=48.80, azimuth=150.00
TX tx_12 placed at x=-979.14, y=904.75, z=48.80, azimuth=280.00
TX tx_13 placed at x=-1004.85, y=791.78, z=48.80, azimuth=330.00
TX tx_14 placed at x=-1001.16, y=780.47, z=48.80, azimuth=90.00
TX tx_15 placed at x=-1002.76, y=769.32, z=48.80, azimuth=210.00
TX tx_16 placed at x=1803.01, y=824.80, z=34

In [ ]:
# Approach 1: Compute Radio Map on a flat bounding grid solver

rm_solver = RadioMapSolver()

rm = rm_solver(scene,
    max_depth=5,           # Maximum number of ray-scene interactions
    samples_per_tx=10**7 , # Total Monte Carlo ray samples per Tx (higher = less noise, more memory)
    cell_size=(5, 5),      # Spatial resolution of each radio map grid cell (in meters)
    center=[0, 0, 0],      # Coordinate center point of the target radio map grid
    size=[5000, 5000],     # Total dimensions (width, height, in meters) of the radio map area
    orientation=[0, 0, 0]) # Spatial orientation of the evaluation grid planes


In [ ]:
# Visualize the calculated propagation metrics
rm.show(metric="path_gain")

# Visualize the Received Signal Strength (RSS)
rm.show(metric="rss")

# Visualize the Signal-to-Interference-plus-Noise Ratio (SINR)
rm.show(metric="sinr")

In [10]:
# Approach 2: Compute Radio Map adaptively conforming to the continuous 3D terrain surface mesh

rm_solver = RadioMapSolver()

# Extract and duplicate the terrain surface as the continuous measuring mesh target
mesurement_surface = scene.objects["terrain"].clone(as_mesh=True)

rm = rm_solver(scene,
    measurement_surface=mesurement_surface,
    samples_per_tx=10**8,   # Increased ray counts for complex surface conformal calculations
    max_depth=5)

In [ ]:
# Debug Block: Manually verify single arbitrary coordinate mapping and placement
x, y, z = latlonh_to_xyz_batch(48.72306, 2.28583, 100)
add_tx(scene, 'tx', (x, y, z), (0, 0, 1))

In [ ]:
# Debug Block: Clean up the temporary test node from the simulation scene
scene.remove('tx')

In [15]:
# Render interactive previews of the 3D scene topology
scene.preview()

In [11]:
# Preview the 3D scene overlaid with the computed Radio Map coverage
scene.preview(radio_map=rm, rm_vmin=-200)

In [ ]:
# Debug Block: Inspect and verify all loaded objects within the active Sionna scene structure
scene.objects

In [ ]:
# Debug Block: Verify electromagnatic property bindings of specified objects
scene.get("water").radio_material